In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
!pip3 install tiktoken
import tiktoken
import tensorflow as tf
import tqdm
from torch.utils.data import Dataset,DataLoader

In [ ]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
import sys
sys.path.append('/kaggle/input/datasets/adnanik23/inputs-gpt-2')
from gpt_download3 import download_and_load_gpt2

In [ ]:
with open('/kaggle/input/datasets/adnanik23/inputs-gpt-2/the-verdict.txt','r',encoding='utf-8') as f:
  document=f.read()

In [ ]:
class MultiHeadAttention(nn.Module):
  def __init__(self,context_length,embed_dim,d_out,num_heads,dropout,qkv_bias=False):
    super().__init__()
    self.d_out=d_out
    self.head_dim=d_out//num_heads
    self.num_heads=num_heads
    self.w_query=nn.Linear(embed_dim,d_out,bias=qkv_bias)
    self.w_key=nn.Linear(embed_dim,d_out,bias=qkv_bias)
    self.w_value=nn.Linear(embed_dim,d_out,bias=qkv_bias)
    self.dropout=nn.Dropout(dropout)
    self.register_buffer("mask",torch.triu(torch.ones(context_length,context_length),diagonal=1))
  def forward(self,x):
    b,num_tokens,embed_shape=x.shape
    query=self.w_query(x)
    key=self.w_key(x)
    value=self.w_value(x)       ### (b,num_tokens,d_out)###

    query=query.view(b,num_tokens,self.num_heads,self.head_dim)
    key=key.view(b,num_tokens,self.num_heads,self.head_dim)
    value=value.view(b,num_tokens,self.num_heads,self.head_dim)       ###(b,num_tokens,num_heads,head_dim)###
    
    query=query.transpose(1,2)
    key=key.transpose(1,2)
    value=value.transpose(1,2)          ###(b,num_heads,num_tokens,head_dim)
    
    attn_scores=query@key.transpose(2,3)
    attn_scores.masked_fill(self.mask.bool()[:num_tokens,:num_tokens],-torch.inf)
    attn_weights=torch.softmax((attn_scores/(self.head_dim**0.5)),dim=-1)
    attn_weights=self.dropout(attn_weights)              
    context_vec=attn_weights@value                ### (b,num_heads,num_tokens,num_tokens)@(b,num_heads,num_tokens,head_dim)
    
    context_vec=context_vec.transpose(1,2)                    ###(b,num_tokens,num_heads,head_dim)
    
    context_vec=context_vec.contiguous().view(b,num_tokens,self.d_out)
    return context_vec

In [ ]:
class LayerNorm(nn.Module):
  def __init__(self,embed_dim,eps=1e-5):
    super().__init__()
    self.eps=eps
    self.scale=nn.Parameter(torch.ones(embed_dim))
    self.shift=nn.Parameter(torch.zeros(embed_dim))
  def forward(self,x):
    mean=x.mean(dim=-1,keepdim=True)
    variance=x.var(dim=-1,unbiased=False,keepdim=True)
    norm_x=(x-mean)/torch.sqrt(variance+self.eps)
    return self.scale*norm_x + self.shift

In [ ]:
class GeLU(nn.Module):
  def __init__(self):
    super().__init__()
  def forward(self,x):
    return nn.functional.gelu(x)

In [ ]:
class FFNN(nn.Module):
  def __init__(self,embed_dim):
    super().__init__()
    self.network=nn.Sequential(
        nn.Linear(embed_dim,4*embed_dim),
        GeLU(),
        nn.Linear(4*embed_dim,embed_dim)
    )
  def forward(self,x):
    return self.network(x)

In [ ]:
class TransformerBlock(nn.Module):
  def __init__(self,config):
    super().__init__()
    self.attn=MultiHeadAttention(context_length=config["context_length"],embed_dim=config["embed_dim"],d_out=config["embed_dim"],num_heads=config["num_heads"],dropout=config["dropout"],qkv_bias=config["qkv_bias"])
    self.layer_norm1=LayerNorm(embed_dim=config["embed_dim"])
    self.layer_norm2=LayerNorm(embed_dim=config["embed_dim"])
    self.neural_net=FFNN(embed_dim=config["embed_dim"])
    self.dropout=nn.Dropout(config["dropout"])
  
  def forward(self,x):
    shortcut=x
    x=self.layer_norm1(x)
    x=self.attn(x)
    x=self.dropout(x)
    x=shortcut+x              ##Shortcut Connection
    shortcut=x
    x=self.layer_norm2(x)
    x=self.neural_net(x)
    x=self.dropout(x)
    x=shortcut+x              ###Shortcut Connection

    return x


In [ ]:
class GPTModel(nn.Module):
  def __init__(self,config):
    super().__init__()
    self.token_embedding=nn.Embedding(config["vocab_size"],config["embed_dim"])
    self.pos_embedding=nn.Embedding(config["context_length"],config["embed_dim"])
    self.dropout=nn.Dropout(config["dropout"])
    self.trf_blocks=nn.Sequential(*[TransformerBlock(config) for _ in range(config["n_layers"])])
    self.final_layer_norm=LayerNorm(config["embed_dim"])
    self.final_nn=nn.Linear(config["embed_dim"],config["vocab_size"],bias=False)
  def forward(self,x):
    batch,seq_length=x.shape
    token=self.token_embedding(x)
    pos=self.pos_embedding(torch.arange(seq_length,device=x.device))
    input_embedding=token+pos
    input_embedding=self.dropout(input_embedding)
    input_embedding=self.trf_blocks(input_embedding)
    input_embedding=self.final_layer_norm(input_embedding)
    logits=self.final_nn(input_embedding)
    return logits



In [ ]:
def text_to_token_ids(text,tokenizer):
  encoded_ids=tokenizer.encode(text,allowed_special={'<|endoftext|>'})
  return torch.tensor(encoded_ids).unsqueeze(0)
def token_ids_to_text(token_ids,tokenizer):
  flat = token_ids.squeeze(0)
  return tokenizer.decode(flat.tolist())

In [ ]:
def generate_new_text(model,context_length,max_new_tokens,input_idx,top_k=None,temperature=0.0,eos_id=None):
  for _ in range(max_new_tokens):
    input_cond=input_idx[:,-context_length:]      #(b,num_tokens)
    with torch.no_grad():
      logits=model(input_cond)           #(b,num_tokens,vocab_size)
    last_output_vec=logits[:,-1,:]
    if top_k is not None:
      top_logits,_=torch.topk(last_output_vec,top_k)
      masked_logits=torch.where(last_output_vec<top_logits[:,-1].unsqueeze(1),torch.tensor(float('-inf'),device=last_output_vec.device),last_output_vec)
    else:
        masked_logits=last_output_vec
    if temperature>0.0:
      masked_logits=masked_logits/temperature
      probs=torch.softmax(masked_logits,dim=-1)
      output_token=torch.multinomial(probs,num_samples=1)
    else:
      probs=torch.softmax(last_output_vec,dim=-1)    #(b,vocab_size)
      output_token=torch.argmax(probs,dim=-1,keepdim=True)      #(b,1)
    if output_token==eos_id:
      break
    input_idx=torch.cat((input_idx,output_token),dim=1)     #(b,num_tokens+1)
  return input_idx

In [ ]:
settings,params=download_and_load_gpt2(model_size="124M",models_dir="gpt2")

In [ ]:
config={"vocab_size":50257,"context_length":256,"embed_dim":768,"num_heads":12,"n_layers":12,"dropout":0.1,"qkv_bias":False}
new_config=config.copy()

In [ ]:
new_config.update({"context_length":1024,"qkv_bias":True})
new_model=GPTModel(new_config)
new_model.to(device)
new_model.eval()

In [ ]:
def assign(left, right):
    if left.shape != right.shape:
        raise ValueError(f"Shape mismatch. Left: {left.shape}, Right: {right.shape}")
    new_data = torch.tensor(right, dtype=left.dtype, device=left.device)
    # Assign the new data to the .data attribute of the existing parameter
    left.data.copy_(new_data)
    return left

In [ ]:
def load_weights_into_model(model,params):
  model.token_embedding.weight=assign(model.token_embedding.weight,params["wte"])
  model.pos_embedding.weight=assign(model.pos_embedding.weight,params["wpe"])
  for i in range(len(params["blocks"])):
    q_w,k_w,v_w=np.split(params['blocks'][i]['attn']['c_attn']['w'],3,axis=-1)
    model.trf_blocks[i].attn.w_query.weight=assign(model.trf_blocks[i].attn.w_query.weight,q_w.T)
    model.trf_blocks[i].attn.w_key.weight=assign(model.trf_blocks[i].attn.w_key.weight,k_w.T)
    model.trf_blocks[i].attn.w_value.weight=assign(model.trf_blocks[i].attn.w_value.weight,v_w.T)

    q_b,k_b,v_b=np.split(params['blocks'][i]['attn']['c_attn']['b'],3,axis=-1)
    model.trf_blocks[i].attn.w_query.bias=assign(model.trf_blocks[i].attn.w_query.bias,q_b)
    model.trf_blocks[i].attn.w_key.bias=assign(model.trf_blocks[i].attn.w_key.bias,k_b)
    model.trf_blocks[i].attn.w_value.bias=assign(model.trf_blocks[i].attn.w_value.bias,v_b)

    model.trf_blocks[i].neural_net.network[0].weight=assign(model.trf_blocks[i].neural_net.network[0].weight,params["blocks"][i]["mlp"]["c_fc"]["w"].T)
    model.trf_blocks[i].neural_net.network[0].bias=assign(model.trf_blocks[i].neural_net.network[0].bias,params["blocks"][i]["mlp"]["c_fc"]["b"])
    model.trf_blocks[i].neural_net.network[2].weight=assign(model.trf_blocks[i].neural_net.network[2].weight,params["blocks"][i]["mlp"]["c_proj"]["w"].T)
    model.trf_blocks[i].neural_net.network[2].bias=assign(model.trf_blocks[i].neural_net.network[2].bias,params["blocks"][i]["mlp"]["c_proj"]["b"])

    model.trf_blocks[i].layer_norm1.scale=assign(model.trf_blocks[i].layer_norm1.scale,params['blocks'][i]['ln_1']['g'])
    model.trf_blocks[i].layer_norm1.shift=assign(model.trf_blocks[i].layer_norm1.shift,params['blocks'][i]['ln_1']['b'])
    model.trf_blocks[i].layer_norm2.scale=assign(model.trf_blocks[i].layer_norm2.scale,params['blocks'][i]['ln_2']['g'])
    model.trf_blocks[i].layer_norm2.shift=assign(model.trf_blocks[i].layer_norm2.shift,params['blocks'][i]['ln_2']['b'])
  
  model.final_layer_norm.scale=assign(model.final_layer_norm.scale,params['g'])
  model.final_layer_norm.shift=assign(model.final_layer_norm.shift,params['b'])
  model.final_nn.weight=assign(model.final_nn.weight,params['wte'])

In [ ]:
load_weights_into_model(new_model,params)

In [ ]:
tokenizer=tiktoken.get_encoding('gpt2')
token_ids=generate_new_text(model=new_model,input_idx=text_to_token_ids("Every effort moves you",tokenizer).to(device),max_new_tokens=25,
                            context_length=new_config["context_length"],top_k=50,temperature=1.5)

In [ ]:
output=token_ids_to_text(token_ids,tokenizer)

In [ ]:
output